# 🚀 Phase 3: The Transformer Bosst (BERT Fine-Tuning)

## 1. Setup & Environment (Colab GPU Recommended)

In [1]:
import os
import sys

# 1. Environment Detection
if 'google.colab' in sys.modules:
    !pip install -q transformers[torch] datasets evaluate accelerate umap-learn
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/Project 2/project-nlp-challenge'
    print(f"Colab GPU Environment. BASE_PATH: {BASE_PATH}")
else:
    BASE_PATH = '.'
    print("💻 Local Environment. Note: Training will be slow without GPU.")

import pandas as pd
import numpy as np
import torch
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
Mounted at /content/drive
Colab GPU Environment. BASE_PATH: /content/drive/MyDrive/Project 2/project-nlp-challenge


## 2. Data Loading & Preparation
We load our cleaned dataset and convert it into the HuggingFace `Dataset` format.

In [5]:
# Load cleaned data (from Phase 1)
df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/cleaned_data.csv'))
df['full_text'] = df['full_text'].fillna('')

# We only need 'full_text' and 'label'
# Note: We take a sample if we want to test the pipeline quickly (e.g., sample=None for full training)
dataset_full = Dataset.from_pandas(df[['full_text', 'label']])
dataset_split = dataset_full.train_test_split(test_size=0.2, seed=42)

print(f"Dataset loaded. Training size: {len(dataset_split['train'])}, Test size: {len(dataset_split['test'])}")

Dataset loaded. Training size: 31908, Test size: 7978


## 3. Tokenization (DistilBERT)
Transformers don't use simple regex cleaning; they use **Subword Tokenization** (WordPiece). This handles out-of-vocabulary words better than Word2Vec.

In [6]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Truncate to 512 tokens (BERT limit)
    return tokenizer(examples["full_text"], padding="max_length", truncation=True)

tokenized_datasets = dataset_split.map(tokenize_function, batched=True)
print("Tokenization complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/31908 [00:00<?, ? examples/s]

Map:   0%|          | 0/7978 [00:00<?, ? examples/s]

Tokenization complete.


## 4. Fine-Tuning Management
We use the HuggingFace `Trainer` API to fine-tune our model for 2 or 3 epochs.

## 5. Load Pre-trained Model (Optional, if already trained)

If the model has already been trained and saved, we can load it directly to skip the training step and proceed with evaluation or prediction.

In [7]:
# Load pre-trained model and tokenizer if available
final_model_path = os.path.join(BASE_PATH, 'models/distilbert_classifier')

if os.path.exists(final_model_path):
    print(f"Cargando modelo y tokenizer de: {final_model_path}")
    model = AutoModelForSequenceClassification.from_pretrained(final_model_path)
    tokenizer = AutoTokenizer.from_pretrained(final_model_path)
else:
    print("No se encontró un modelo pre-entrenado. Asegúrate de ejecutar la celda de entrenamiento si no existe.")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Re-initialize the Trainer with the (potentially loaded) model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

Cargando modelo y tokenizer de: /content/drive/MyDrive/Project 2/project-nlp-challenge/models/distilbert_classifier


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

NameError: name 'training_args' is not defined

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir=os.path.join(BASE_PATH, "models/bert_results"),
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_total_limit=1,
    report_to="none"
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 5. Train and Save the new model as distilbert_classifier

In [ ]:

# INICIAR ENTRENAMIENTO (Necesitas GPU en Colab)
trainer.train()

# 💾 GUARDAR MODELO FINAL PARA GRÁFICAS Y COMPARACIÓN
final_model_path = os.path.join(BASE_PATH, 'models/distilbert_classifier')
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"Modelo guardado en: {final_model_path}")

Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy
1,0.000399,0.006338,0.002300,0.999248
2,0.000048,0.001172,0.002300,0.999875


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en: /content/drive/MyDrive/Project 2/project-nlp-challenge/models/distilbert_classifier


## 7. Final Delivery: Predictions on Validation Data
We use our best model to generate the final competition file.

In [9]:
def generate_predictions(val_csv_path, output_csv_path, trainer, tokenizer_func):
    """
    Enhanced prediction generator:
    - Injects trainer/tokenizer to avoid global state issues.
    - Prunes memory by removing raw text after tokenization.
    - Validates schema and ensures output directories exist.
    """
    # 0. Safety: Check if input exists and output directory is ready
    if not os.path.exists(val_csv_path):
        raise FileNotFoundError(f"Input file not found: {val_csv_path}")

    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)

    # 1. Load and Validate Schema
    val_df = pd.read_csv(val_csv_path)
    required_cols = ['title', 'text']
    for col in required_cols:
        if col not in val_df.columns:
            raise KeyError(f"Missing required column '{col}' in {val_csv_path}")
    # 2. Preprocess with NaN protection
    val_df['full_text'] = val_df['title'].fillna('') + " " + val_df['text'].fillna('')
    # 🏆 CARGAMOS EL MODELO GANADOR (99.9% accuracy) DE LA CARPETA DE CHECKPOINTS
    best_path = os.path.join(BASE_PATH, 'models/bert_results/checkpoint-1500')
    print(f'🎖 Cargando pesos ganadores de: {best_path}')
    best_model = AutoModelForSequenceClassification.from_pretrained(best_path)
    # Creamos un trainer específico para estas predicciones finales
    prediction_trainer = Trainer(model=best_model)

    # 3. Memory-Efficient Tokenization
    # We remove 'full_text' after tokenization to save Colab RAM
    val_dataset = Dataset.from_pandas(val_df[['full_text']])
    tokenized_val = val_dataset.map(
        tokenizer_func,
        batched=True,
        remove_columns=val_dataset.column_names  # Prune memory!
    )

    # 4. Predict (High-performance inference)
    print("Running inference on BERT...")
    raw_preds = prediction_trainer.predict(tokenized_val)
    predictions = np.argmax(raw_preds.predictions, axis=-1)

    # 5. Assemble results
    val_df['label'] = predictions
    ideal_columns = ['label', 'title', 'text', 'subject', 'date']
    final_columns = [col for col in ideal_columns if col in val_df.columns]

    # 6. Export safely
    val_df[final_columns].to_csv(output_csv_path, index=False, sep=',')
    print(f"Final Deliverable generated at: {output_csv_path}")
# --- Execution Example ---
# generate_predictions(valid_path, final_path, trainer, tokenize_function)

In [3]:
valid_path = os.path.join(BASE_PATH, 'dataset/validation_data.csv')
final_path = os.path.join(BASE_PATH, 'dataset/validation_results_bert.csv')

In [ ]:
"""def generate_predictions(val_csv_path, output_csv_path):
    val_df = pd.read_csv(val_csv_path)

    # 1. Preprocess validation text (Title + Text)
    # ⚠️ Prevención ante NaNs: rellenamos con texto vacío antes de concatenar
    val_df['full_text'] = val_df['title'].fillna('') + " " + val_df['text'].fillna('')
    # 🏆 CARGAMOS EL MODELO GANADOR (99.9% accuracy) DE LA CARPETA DE CHECKPOINTS
    best_path = os.path.join(BASE_PATH, 'models/bert_results/checkpoint-1500')
    print(f'🎖 Cargando pesos ganadores de: {best_path}')
    best_model = AutoModelForSequenceClassification.from_pretrained(best_path)
    # Creamos un trainer específico para estas predicciones finales
    prediction_trainer = Trainer(model=best_model)

    # Note: For BERT, we use raw text to allow BERT to capture punctuation and context.
    val_dataset = Dataset.from_pandas(val_df[['full_text']])
    tokenized_val = val_dataset.map(tokenize_function, batched=True)

    # 2. Predict (Trainer handles GPU batching)
    raw_preds = prediction_trainer.predict(tokenized_val)
    predictions = np.argmax(raw_preds.predictions, axis=-1)

    # 3. Replace labels with predictions
    val_df['label'] = predictions

    # 4. Filter and Export safely
    ideal_columns = ['label', 'title', 'text', 'subject', 'date']
    final_columns = [col for col in ideal_columns if col in val_df.columns]

    val_df[final_columns].to_csv(output_csv_path, index=False, sep=',')
    print(f"\Final Deliverable generated at: {output_csv_path}")

valid_path = os.path.join(BASE_PATH, 'dataset/validation_data.csv')
final_path = os.path.join(BASE_PATH, 'dataset/validation_results_bert.csv')

"""

In [10]:
# FINAL EXECUTION: Generate the deliverable for the competition
# -------------------------------------------------------------------------------------


if os.path.exists(valid_path):

    generate_predictions(valid_path, final_path, trainer, tokenize_function)
    print(f"Final deliverable is ready at: {final_path}")
else:
    print(f"Validation file not found at {valid_path}. Check your dataset folder!")

🎖 Cargando pesos ganadores de: /content/drive/MyDrive/Project 2/project-nlp-challenge/models/bert_results/checkpoint-1500


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Map:   0%|          | 0/4956 [00:00<?, ? examples/s]

Running inference on BERT...


Final Deliverable generated at: /content/drive/MyDrive/Project 2/project-nlp-challenge/dataset/validation_results_bert.csv
Final deliverable is ready at: /content/drive/MyDrive/Project 2/project-nlp-challenge/dataset/validation_results_bert.csv
